In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/cleaned/fraud_features.csv")
df["Recent_Address_Change"] = df["AddressChange_Claim"].isin(["under 6 months", "2 to 3 years"]).astype(int)

feature_cols_num = ["Age_Clean", "VehiclePrice_ord", "PastNumberOfClaims_ord", "AgeOfVehicle_ord",
                     "NumberOfSuppliments_ord", "Days_Policy_Accident_ord", "Days_Policy_Claim_ord",
                     "Claim_Delay_Months", "DriverRating", "Deductible", "Year"]
feature_cols_cat = ["PolicyType", "Fault", "AccidentArea", "VehicleCategory", "Sex", "MaritalStatus",
                     "AgentType", "BasePolicy"]
feature_cols_bin = ["Police_Report", "Witness", "Policyholder_At_Fault", "Agent_External",
                     "Weekend_Accident", "Weekend_Claim", "Age_Missing", "Recent_Address_Change"]
feature_cols = feature_cols_num + feature_cols_cat + feature_cols_bin

X = df[feature_cols].copy()
y = df["FraudFound_P"].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
    ("bin", "passthrough", feature_cols_bin),
])

rf = Pipeline([("pre", preprocessor),
               ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                        class_weight="balanced", random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]
print("Test PR-AUC:", round(average_precision_score(y_test, proba), 3))
print("Test ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

Test PR-AUC: 0.207
Test ROC-AUC: 0.82


In [2]:
policy_priority = pd.DataFrame({
    "PolicyType": ["Sport - Collision", "Utility - All Perils", "Sedan - All Perils",
                   "Sedan - Collision", "Sedan - Liability"],
    "Fraud_Rate_%": [13.79, 12.06, 10.06, 6.88, 0.72],
    "N_Claims": [348, 340, 4086, 5584, 4987]
}).sort_values("Fraud_Rate_%", ascending=False)
print(policy_priority)

             PolicyType  Fraud_Rate_%  N_Claims
0     Sport - Collision         13.79       348
1  Utility - All Perils         12.06       340
2    Sedan - All Perils         10.06      4086
3     Sedan - Collision          6.88      5584
4     Sedan - Liability          0.72      4987


In [3]:
n_test = len(y_test)
actual_fraud_in_test = y_test.sum()   # 185

capacity_levels = [5, 10, 15, 20, 25, 30, 40, 50]
results = []
for pct in capacity_levels:
    cutoff = np.percentile(proba, 100 - pct)
    flagged = proba >= cutoff
    fraud_caught = y_test[flagged].sum()
    results.append({
        "Capacity_%": pct,
        "Claims_Flagged": flagged.sum(),
        "Fraud_Caught": fraud_caught,
        "Capture_Rate_%": round(100 * fraud_caught / actual_fraud_in_test, 1),
        "Precision_%": round(100 * y_test[flagged].mean(), 1),
    })
capacity_table = pd.DataFrame(results)
print(capacity_table)

   Capacity_%  Claims_Flagged  Fraud_Caught  Capture_Rate_%  Precision_%
0           5             155            41            22.2         26.5
1          10             309            63            34.1         20.4
2          15             463            86            46.5         18.6
3          20             617           104            56.2         16.9
4          25             771           122            65.9         15.8
5          30             925           137            74.1         14.8
6          40            1234           163            88.1         13.2
7          50            1542           181            97.8         11.7


In [4]:
# Compare model-based selection to random selection at 20% capacity (a stronger operating point)
capacity_pct = 20
fraud_caught_model = 104          # from your table
random_expected = 185 * (capacity_pct / 100)   # 37.0

avg_claim_cost = 8000
print(f"Model-prioritized (top 20%): {fraud_caught_model} fraud cases caught "
      f"(~${fraud_caught_model*avg_claim_cost:,.0f})")
print(f"Random 20% audit (expected): {random_expected:.1f} fraud cases "
      f"(~${random_expected*avg_claim_cost:,.0f})")
print(f"Uplift: {fraud_caught_model - random_expected:.1f} more cases caught "
      f"({(fraud_caught_model/random_expected - 1)*100:.0f}% more than random)")

Model-prioritized (top 20%): 104 fraud cases caught (~$832,000)
Random 20% audit (expected): 37.0 fraud cases (~$296,000)
Uplift: 67.0 more cases caught (181% more than random)


## Business Recommendation

**Decision:** Deploy the Random Forest fraud-risk score (PR-AUC 0.207, 5-fold CV 0.193±0.013)
to rank incoming claims for SIU investigator review, replacing random/rule-based selection.

**Priority segments (Step 4/5):**
- Highest-risk policy types: Sport-Collision (13.79% fraud), Utility-All Perils (12.06%)
- Lowest-risk: Sedan-Liability (0.72% fraud)
- Top model drivers: PolicyType, BasePolicy, Fault, VehicleCategory, Policyholder-at-fault status

**Operational recommendation:** investigate the top 15-20% of claims by model score.
At 20% capacity, the model catches 104 of 185 fraud cases (56.2% recall) at 16.9% precision,
nearly 3x better than the dataset's 6% base rate. Beyond 30% capacity, returns flatten sharply
(precision falls toward the random baseline), so this range is where investigator effort
should concentrate.

**Estimated impact:** at 20% capacity, model-based prioritization catches an estimated 67 more
fraud cases than random selection at the same review volume (181% more), worth roughly
$536,000 in additional fraud caught using an illustrative $8,000 average claim cost
(a placeholder — should be validated against real claim-payout data before use).

**Model performance:** Random Forest, PR-AUC 0.207 (test), 0.193±0.013 (5-fold CV),
ROC-AUC 0.820. Accuracy is not used as a headline metric — with 5.99% fraud, a model
predicting "no fraud" for every claim would score 94.0% accuracy while catching zero fraud.

**Risks and limitations:**
- Data is from 1994-1996 — fraud patterns and claim costs have likely shifted substantially;
  retraining on current claims is required before real deployment
- Dollar-impact figures use an illustrative average claim cost, not real payout data
- Flagged claims require human investigation, not automatic denial
- False positives carry real costs (investigator time, customer trust); the 15-20% capacity
  range should be validated against the SIU team's actual bandwidth